# WP38 — Analogical Reasoning Engine (v0.4)
## RelationalGraph · StructureMapper · AnalogyEngine

Demonstrates **WP38**: Hofstadterian structure-mapping — transferring policy weights across domains (Chess → Go → Synthesis) via relational isomorphism, without target-domain labels.

> *"The meaning of a symbol is not in the symbol itself, but in its place within the system."* — Hofstadter (1979)

Runtime: **~2 min**

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')); 
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np, matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp38_analogy_engine import (
    RelationalGraph, StructureMapper, AnalogyScore, AnalogyMap,
    TransferPolicy, AnalogyEngine,
    make_chess_graph, make_go_graph, make_synthesis_graph,
    verify_wp38_exit_criteria
)
chess = make_chess_graph()
go    = make_go_graph()
synth = make_synthesis_graph()
print(f'Chess graph:     {len(chess.entities)} entities, {len(chess.relations)} relations')
print(f'Go graph:        {len(go.entities)} entities, {len(go.relations)} relations')
print(f'Synthesis graph: {len(synth.entities)} entities, {len(synth.relations)} relations')

In [ ]:
print('Chess entities and their degrees:')
for e in chess.entities:
    deg = chess.degree(e)
    rels = [r.relation_type for r in chess.relations_of_entity(e)]
    print(f'  {e:<18} degree={deg}  relations={list(set(rels))}')

In [ ]:
mapper = StructureMapper(seed=42)
amap = mapper.map(chess, go)
print('Chess -> Go Analogy Map:')
print(f'  Entity score:   {amap.score.entity_score:.3f}')
print(f'  Relation score: {amap.score.relation_score:.3f}')
print(f'  Overall score:  {amap.score.overall:.3f}')
print()
print('Entity Mapping (Chess entity -> Go analogue):')
for src, tgt in sorted(amap.mapping.items()):
    print(f'  {src:<20} -> {tgt}')

In [ ]:
# TransferPolicy: copy chess weights to go domain via analogy (no labels)
policy = TransferPolicy(fallback='mean')
chess_weights = {e: (i+1)/len(chess.entities) for i, e in enumerate(sorted(chess.entities))}
go_weights    = policy.transfer(amap, chess_weights, go.entities)
print('Source (Chess) policy weights:')
for e, w in sorted(chess_weights.items()):
    print(f'  {e:<20} {w:.3f}')
print()
print('Target (Go) policy weights — transferred via analogy:')
for e, w in sorted(go_weights.items()):
    marker = ' <- mapped' if amap.mapping.get(e) or any(v==e for v in amap.mapping.values()) else ''
    print(f'  {e:<22} {w:.3f}{marker}')

In [ ]:
# Full AnalogyEngine pipeline: Chess -> Go, Chess -> Synthesis
engine = AnalogyEngine()
src_w  = {e: 1/len(chess.entities) for e in chess.entities}
go_w,    rec1 = engine.transfer(chess, go,    src_w)
synth_w, rec2 = engine.transfer(chess, synth, src_w)
print('AnalogyEngine transfer summary:')
for rec in [rec1, rec2]:
    d = rec.to_dict()
    print(f'  {d["source_domain"]}->{d["target_domain"]}:  score={d["analogy_score"]["overall"]:.3f}'
          f'  mapped={d["n_mapped_entities"]}/{d["n_source_entities"]}  preserved_relations={d["n_preserved_relations"]}')

In [ ]:
import matplotlib.patches as mpatches
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: analogy score breakdown
ax = axes[0]
transfers = ['Chess->Go', 'Chess->Synthesis']
scores = [rec1.analogy_score, rec2.analogy_score]
for i, (label, score) in enumerate(zip(transfers, scores)):
    ax.bar([i-0.2], [score.entity_score], 0.35, label='Entity score' if i==0 else '', color='#2196F3', alpha=0.85)
    ax.bar([i+0.2], [score.relation_score], 0.35, label='Relation score' if i==0 else '', color='#FF9800', alpha=0.85)
ax.set_xticks(range(2)); ax.set_xticklabels(transfers)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.2)
ax.set_title('Analogy Scores per Transfer', fontweight='bold'); ax.legend()

# Panel B: Chess -> Go weight mapping
ax2 = axes[1]
chess_sorted = sorted(chess_weights.items(), key=lambda x: -x[1])
bar_labels = [f'{src}\n->\n{amap.mapping.get(src,"?")[:10]}' for src, _ in chess_sorted[:6]]
ax2.bar(range(len(chess_sorted[:6])), [w for _,w in chess_sorted[:6]], color='#4CAF50', edgecolor='black', alpha=0.85)
ax2.set_xticks(range(6)); ax2.set_xticklabels(bar_labels, fontsize=8)
ax2.set_ylabel('Weight'); ax2.set_title('Chess Weights (with Go analogues)', fontweight='bold')

# Panel C: degree correlation
ax3 = axes[2]
reverse = {v:k for k,v in amap.mapping.items()}
go_ents_mapped = [(e, chess.degree(reverse[e]) if e in reverse else 0, go.degree(e)) for e in go.entities if e in reverse]
ax3.scatter([x[1] for x in go_ents_mapped], [x[2] for x in go_ents_mapped], s=80, color='#9C27B0', zorder=3)
for name, cd, gd in go_ents_mapped:
    ax3.annotate(name[:8], (cd, gd), fontsize=7)
ax3.set_xlabel('Chess entity degree'); ax3.set_ylabel('Go entity degree')
ax3.set_title('Degree Correlation: Mapped Entities', fontweight='bold')

fig.suptitle('WP38: Analogical Reasoning Engine', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp38_analogy_engine.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved wp38_analogy_engine.png')

In [ ]:
criteria = verify_wp38_exit_criteria(engine, [rec1, rec2])
print('WP38 Exit Criteria Verification'); print('='*65)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()): print('\nAll WP38 exit criteria satisfied.')

---
## Conclusions

**WP38** implements Hofstadterian structure mapping:
- `RelationalGraph` encodes domain structure as entities + typed relations
- `StructureMapper` finds best-isomorphism via degree + relation-type similarity
- `TransferPolicy` copies source weights to target domain without labels

### References
- Hofstadter (1979) *Gödel, Escher, Bach* — isomorphisms between formal systems
- Gentner (1983) Structure-Mapping Theory
- Good (1965) — cross-domain analogical transfer as a prerequisite for intelligence explosion